# GLUE fine-tuning eval on L4
Sets up babylm-eval fresh and runs GLUE fine-tuning, much faster than local M1.
Runtime > Change runtime type > L4 GPU, before running this.

In [1]:
!nvidia-smi

Mon Jul 27 01:43:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   32C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!rm -rf babylm-eval
!git clone https://github.com/babylm-org/babylm-eval.git
import os
print(os.listdir("babylm-eval"))

Cloning into 'babylm-eval'...
remote: Enumerating objects: 5915, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 5915 (delta 80), reused 67 (delta 67), pack-reused 5814 (from 2)
Receiving objects: 100% (5915/5915), 44.11 MiB | 21.76 MiB/s, done.
Resolving deltas: 100% (3714/3714), done.
['multilingual', 'strict', '.gitignore', 'requirements.txt', 'README.md', '.git']


In [3]:
os.chdir("babylm-eval/strict")
!pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.2.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.3 which is incompatible.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.7.0 which is incompatible.
google-genai 2.11.0 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.11.4 which is incompatible.
google-genai 2.11.0 requires typing-extensions<5.0.0,>=4.14.0, but you have typing-extensions 4.13.2 which is incompatible.
openai 2.45.0 requires typing-extensions<5,>=4.14, but you have typing-extensions 4.13.2 which is incompatible.
google-adk 2.4.0 requires pydantic<3,>=2.12, but you have pydantic 2.11.4 which is incompatible.
goog

In [5]:
!python -m scripts.download_evals

Fetching 175 files:   0% 0/175 [00:00<?, ?it/s]
complex_NP_island.jsonl: 0.00B [00:00, ?B/s]

complex_NP_island.jsonl: 78.3kB [00:00, 43.0MB/s]
causative.jsonl: 63.0kB [00:00, 67.1MB/s]

.gitattributes: 2.69kB [00:00, 613kB/s]
Fetching 175 files:   1% 1/175 [00:00<01:00,  2.89it/s]
animate_subject_passive.jsonl: 67.5kB [00:00, 42.8MB/s]

anaphor_number_agreement.jsonl: 66.1kB [00:00, 122MB/s]

adjunct_island.jsonl: 71.5kB [00:00, 29.0MB/s]

animate_subject_trans.jsonl: 64.0kB [00:00, 22.2MB/s]

anaphor_gender_agreement.jsonl: 64.6kB [00:00, 117MB/s]
Fetching 175 files:   2% 3/175 [00:00<00:28,  6.01it/s]
(…)ure_constraint_complex_left_branch.jsonl: 80.5kB [00:00, 127MB/s]

(…)erminer_noun_agreement_irregular_1.jsonl: 70.7kB [00:00, 140MB/s]

determiner_noun_agreement_2.jsonl: 69.4kB [00:00, 148MB/s]

(…)erminer_noun_agreement_irregular_2.jsonl: 70.5kB [00:00, 170MB/s]

(…)terminer_noun_agreement_with_adj_2.jsonl: 74.1kB [00:00, 170MB/s]

(…)cture_constraint_object_extraction.jsonl: 73.

In [6]:
# force sequence length to 256 to match our model's n_positions,
# and cut epochs to 2, fine-tuning converges fast, and 10 epochs
# across 7 tasks is not feasible in the time we have
!sed -i 's/--sequence_length 512/--sequence_length 256/g' scripts/eval_finetuning.sh
!sed -i 's/--num_epochs 10/--num_epochs 2/g' scripts/eval_finetuning.sh
!grep -n "sequence_length\|num_epochs" scripts/eval_finetuning.sh

71:        --num_epochs $MAX_EPOCHS \
72:        --sequence_length 256 \
94:    --num_epochs $MAX_EPOCHS \
95:    --sequence_length 256 \
116:    --num_epochs $WSC_EPOCHS \
117:    --sequence_length 256 \
140:        --num_epochs $MAX_EPOCHS \
141:        --sequence_length 256 \
163:    --num_epochs $MAX_EPOCHS \
164:    --sequence_length 256 \


In [9]:
!pip install -q --upgrade torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 40.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 25.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 9.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━

In [7]:
from google.colab import drive
drive.mount('/content/drive')

# upload your model_a/final folder to drive first, at this path
# (adjust if you put it somewhere else)
MODEL_A_PATH = '/content/drive/MyDrive/babylm_checkpoints/model_a/final'
!ls "$MODEL_A_PATH"

Mounted at /content/drive
config.json		model.safetensors      tokenizer.json
generation_config.json	tokenizer_config.json  training_args.bin


In [10]:
!./scripts/eval_finetuning.sh --model_path "$MODEL_A_PATH"

./scripts/eval_finetuning.sh: line 4: ../.env: No such file or directory
boolq
2026-07-27 01:49:32.226820: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-27 01:49:32.449558: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violatio

In [12]:
!zip -r results.zip results/
from google.colab import files
files.download('results.zip')

updating: results/ (stored 0%)
updating: results/final/ (stored 0%)
updating: results/final/main/ (stored 0%)
updating: results/final/main/finetune/ (stored 0%)
updating: results/final/main/finetune/boolq/ (stored 0%)
updating: results/final/main/finetune/boolq/results.txt (deflated 12%)
updating: results/final/main/finetune/boolq/predictions.json (deflated 92%)
updating: results/final/main/finetune/qqp/ (stored 0%)
updating: results/final/main/finetune/qqp/results.txt (deflated 12%)
updating: results/final/main/finetune/qqp/predictions.json (deflated 91%)
updating: results/final/main/finetune/wsc/ (stored 0%)
updating: results/final/main/finetune/wsc/results.txt (deflated 24%)
updating: results/final/main/finetune/wsc/predictions.json (deflated 88%)
updating: results/final/main/finetune/multirc/ (stored 0%)
updating: results/final/main/finetune/multirc/results.txt (deflated 13%)
updating: results/final/main/finetune/multirc/predictions.json (deflated 92%)
updating: results/final/main/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import shutil
shutil.copytree("/content/babylm-eval/strict/results", "/content/drive/MyDrive/babylm_glue_results", dirs_exist_ok=True)

'/content/drive/MyDrive/babylm_glue_results'